# tensor-unbind composite — cx14: unbind per-ray LHS / RHS columns then linalg.solve in batch

> Composite procedural drill from [Delta Drills](https://delta-drills.vercel.app).
> Exercises 2 atoms together: `tensor-unbind`, `linalg-solve-batched`
> Running the final beacon reports progress against all 2 subtopics.

**Why composite drills.** Single-atom drills test atomic skills in isolation. Composite drills test the COMPOSITION — how atoms wire together in real ARENA code. Passing this drill demonstrates you can apply the atoms jointly, not just individually.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)

## Connect to Delta Drills

Paste your Delta Drills auth token below. Beacon will report progress against ALL atoms exercised by this composite.

In [ ]:
# === Delta Drills auth (composite) ===
DD_TOKEN = ""  # paste token, then run
DD_PRIMARY_ATOM = "tensor-unbind"
DD_ATOM_IDS = ["tensor-unbind", "linalg-solve-batched"]
DD_SUBTOPICS = ["Numpy: Indexing and selection", "PyTorch: Batched linalg.solve"]
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"

_dd_passed = set()

## How these two atoms compose

Ray-triangle intersection in ARENA part 1 reduces to a 3x3 linear system per ray:
  `[-D | (B - A) | (C - A)] · [s, u, v]^T = (O - A)`

When you're packing per-ray inputs, it's natural to STACK the three column vectors into a single tensor `cols: (NR, 3, 3)` along axis 2 (each column is one of `-D`, `B-A`, `C-A`). To feed this to `torch.linalg.solve` you don't actually need to split it — `solve` takes the full matrix. But the RHS often arrives stacked too (e.g. multiple offsets per ray), and `unbind` is the canonical way to peel off the slice you want.

This composition pairs `tensor-unbind` (peel a per-ray RHS off a `(NR, K, 3)` block) with `linalg-solve-batched` (solve all NR systems in one batched call). The batched solve is the load-bearing atom — looping over rays would work but is 50-100x slower.

### Composite Exercise — unbind per-ray LHS / RHS columns then linalg.solve in batch

**Atoms exercised together**: `tensor-unbind`, `linalg-solve-batched`

Implement `cx14_solve_per_ray(mats, rhs_stack, which)` that solves a batch of 3x3 linear systems, one per ray.

- `mats: (NR, 3, 3)` — per-ray coefficient matrices.
- `rhs_stack: (NR, K, 3)` — a stack of K per-ray RHS vectors. Axis 1 enumerates which RHS   variant.
- `which: int` — index in `[0, K)` selecting which RHS variant to solve against.

1. **Unbind** `rhs_stack` along axis 1 to get a tuple of K tensors each of shape `(NR, 3)`. Select element `which` from that tuple — this is your per-ray RHS.
2. **Batched solve** with `torch.linalg.solve(mats, rhs)` — the trailing 3-axis of `rhs` is treated as the column vector and broadcasting handles the batch axis. Returns shape `(NR, 3)`.

Cross-check against a Python for-loop calling `torch.linalg.solve` on each ray individually.

In [ ]:
# Fill in the function below, then run this cell. The test asserts the composition is correct.

def cx14_solve_per_ray(mats, rhs_stack, which):
    raise NotImplementedError

def _test_cx14():
    # Case A: small, hand-built — invertible diagonals, known solutions.
    NR, K = 4, 3
    # Per-ray identity-scaled matrices: mats[i] = (i+1) * I_3.
    mats = t.stack([(i + 1.0) * t.eye(3) for i in range(NR)])  # (NR, 3, 3)
    # RHS stack: K=3 variants. Variant 0 = ones; variant 1 = [1,2,3]; variant 2 = [4,5,6].
    v0 = t.ones(NR, 3)
    v1 = t.tensor([[1.0, 2.0, 3.0]]).expand(NR, 3).contiguous()
    v2 = t.tensor([[4.0, 5.0, 6.0]]).expand(NR, 3).contiguous()
    rhs_stack = t.stack([v0, v1, v2], dim=1)  # (NR, 3, 3) — K=3 along axis 1
    assert tuple(rhs_stack.shape) == (NR, K, 3)

    # Pick variant 1 ([1,2,3]). Solution per ray = rhs / (i+1).
    out = cx14_solve_per_ray(mats, rhs_stack, which=1)
    assert tuple(out.shape) == (NR, 3)
    expected = t.stack([t.tensor([1.0, 2.0, 3.0]) / (i + 1.0) for i in range(NR)])
    assert t.allclose(out, expected, atol=1e-5), f'got {out}, expected {expected}'

    # Case B: variant 0 (ones) — solution per ray = 1/(i+1) * ones.
    out0 = cx14_solve_per_ray(mats, rhs_stack, which=0)
    expected0 = t.stack([t.full((3,), 1.0 / (i + 1.0)) for i in range(NR)])
    assert t.allclose(out0, expected0, atol=1e-5)

    # Case C: random invertible mats — cross-check against per-ray loop.
    t.manual_seed(42)
    mats2 = t.randn(6, 3, 3) + 4.0 * t.eye(3)  # well-conditioned
    rhs2 = t.randn(6, 2, 3)
    out2 = cx14_solve_per_ray(mats2, rhs2, which=0)
    ref2 = t.stack([t.linalg.solve(mats2[i], rhs2[i, 0]) for i in range(6)])
    assert t.allclose(out2, ref2, atol=1e-4), f'batched solve diverged from per-ray loop'
    _dd_passed.add('cx14')

_test_cx14()

<details><summary>Show solution — cx14</summary>

```python
def cx14_solve_per_ray(mats, rhs_stack, which):
    # Atom A (tensor-unbind): split the K-axis (axis 1) into a tuple of K tensors
    # each of shape (NR, 3).
    rhs_variants = t.unbind(rhs_stack, dim=1)
    rhs = rhs_variants[which]  # (NR, 3)
    # Atom B (linalg-solve-batched): solve all NR 3x3 systems in one call.
    return t.linalg.solve(mats, rhs)
```

`t.unbind(rhs_stack, dim=1)` returns a Python tuple — indexing `[which]` is a constant-time pick of one stride-view. No copy until `linalg.solve` actually consumes it. For `linalg.solve(mats, rhs)` where `mats: (NR, 3, 3)` and `rhs: (NR, 3)`, PyTorch treats `rhs` as a per-ray column vector (the trailing 3 is the system dimension). If you accidentally pass the full `rhs_stack` (without unbinding), the solver tries to solve K right-hand-sides per ray and you get back `(NR, 3, K)` (or worse, a shape error). Unbind is the explicit-selection step.
</details>

## Report completion

Run the cell below to send progress to Delta Drills. The beacon fires once and reports all 2 subtopics together.

In [ ]:
# === Delta Drills completion beacon (composite — fires for ALL atoms) ===
import urllib.request as _dd_req, json as _dd_json

_DD_REQUIRED = {'cx14'}

def report_completion():
    missing = _DD_REQUIRED - _dd_passed
    if missing:
        print(f"[Delta Drills] {sorted(missing)} not yet passing — fix the cell above, then re-run this one.")
        return
    if not DD_TOKEN:
        print('[Delta Drills] DD_TOKEN is empty — completion not reported.')
        return
    body = _dd_json.dumps({
        'exercise_title': f'composite-drill:{DD_PRIMARY_ATOM}:cx14',
        'subtopics': ["Numpy: Indexing and selection", "PyTorch: Batched linalg.solve"],
        'feedback': 'somewhat',
        'correct': True,
    }).encode('utf-8')
    req = _dd_req.Request(
        f'{DD_BACKEND_URL}/api/practice/arena-rating',
        data=body,
        headers={
            'Content-Type': 'application/json',
            'Authorization': f'Bearer {DD_TOKEN}',
        },
        method='POST',
    )
    try:
        with _dd_req.urlopen(req, timeout=5) as r:
            resp = _dd_json.loads(r.read())
        print(f'[Delta Drills] reported composite (atoms={DD_ATOM_IDS})')
        print(f'[Delta Drills] EWMA updated: {resp}')
    except Exception as e:
        print(f'[Delta Drills] beacon failed: {e}')

report_completion()